# Data Preparation & Visualization

End-to-end preprocessing of the Telco Customer Churn dataset — from raw CSV inspection through cleaning, exploratory visualization, statistical testing, feature engineering, encoding, scaling, and train/test split.

**Dataset:** 7,043 telecom customers with 21 features tracking demographics, services, account info, and churn status.

## Setup

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn import preprocessing, model_selection
import sys
sys.path.insert(0, '.')

In [ ]:
df = pd.read_csv('precleaned-Telco-Customer-Churn.csv')
pd.set_option('display.max_columns', None)
df.head(3)

---
## Task 0 — Describe Data

**Challenge:** Get a quick structural overview of a new dataset without visual inspection.

**New techniques:** `pd.read_csv`, `.shape`, `.dtypes`, `.head()`, `.isna().sum()`, `.duplicated().sum()`

> Key takeaway: The first step of any data project is a high-level structural scan — shape, types, missingness, duplicates — before any cleaning or modeling.

In [ ]:
print('Shape:', df.shape)
print('\nDtypes:')
print(df.dtypes)
print('\nMissing values per column:')
print(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nFirst 3 rows:')
df.head(3)

---
## Task 1 — Plot Missingness

**Challenge:** Visually identify which columns and rows contain missing values, at a glance.

**New techniques:** `np.where(df.isnull().values)`, `plt.scatter(marker="|")`, `plt.yticks(positions, labels)`

> Key takeaway: A missingness scatter plot reveals the shape of incompleteness — vertical streaks indicate whole-column issues, horizontal streaks indicate problematic rows.

In [ ]:
plt.figure(figsize=(12, 8))
row_idx, col_idx = np.where(df.isnull().values)
plt.scatter(row_idx, col_idx, marker='|')
plt.yticks(range(len(df.columns)), df.columns)
plt.title('Missingness Plot')
plt.tight_layout()
plt.show()

---
## Task 2 — Convert Columns

**Challenge:** Fix improperly typed columns — `TotalCharges` stored as string, `SeniorCitizen` as 0/1 that should be 'Yes'/'No'.

**New techniques:** `pd.to_numeric(errors='coerce')`, `.replace()`

> Key takeaway: Data read from CSV often arrives with wrong dtypes; `to_numeric` and `replace` are the primary tools for type correction.

In [ ]:
convert_columns = __import__('2-convert_columns').convert_columns

print('Before — TotalCharges dtype:', df['TotalCharges'].dtype)
print('Before — SeniorCitizen unique:', df['SeniorCitizen'].unique())

df = convert_columns(df)

print('After — TotalCharges dtype:', df['TotalCharges'].dtype)
print('After — SeniorCitizen unique:', df['SeniorCitizen'].unique())

---
## Task 3 — Clean Total Charges

**Challenge:** Handle missing values in `TotalCharges` with three strategies: drop, median fill, formula imputation.

**New techniques:** `dropna(subset=)`, `fillna(median)`, formula imputation, `copy()`

> Key takeaway: Missing value handling is not one-size-fits-all; different imputation strategies have different trade-offs and should be parameterized.

In [ ]:
clean_total_charges = __import__('3-clean_total_charges').clean_total_charges

missing_before = df['TotalCharges'].isna().sum()
print(f'Missing TotalCharges before: {missing_before}')

df = clean_total_charges(df, method='impute')

missing_after = df['TotalCharges'].isna().sum()
print(f'Missing TotalCharges after:  {missing_after}')
print('\nImputation check — first 5 rows with tenure <= 1:')
df[df['tenure'] <= 2][['tenure', 'MonthlyCharges', 'TotalCharges']].head()

---
## Task 4 — Remove Duplicates

**Challenge:** Eliminate redundant identical rows from the DataFrame.

**New techniques:** `drop_duplicates()`

> Key takeaway: Duplicate rows inflate counts and bias statistical tests. One line of pandas fixes it.

In [ ]:
remove_duplicates = __import__('4-remove_duplicates').remove_duplicates

rows_before = len(df)
df = remove_duplicates(df)
print(f'Rows before: {rows_before}')
print(f'Rows after:  {len(df)}')
print(f'Duplicates removed: {rows_before - len(df)}')

---
## Task 5 — Drop customerID

**Challenge:** Remove a column that uniquely identifies each row but has no predictive value.

**New techniques:** `df.drop("col", axis=1)`

> Key takeaway: Unique identifiers must be dropped before modeling — they carry zero signal and can cause data leakage.

In [ ]:
drop_customerID = __import__('5-drop_customerID').drop_customerID

df = drop_customerID(df)
print('Columns after dropping customerID:')
print(df.columns.tolist())

---
## Task 6 — Plot Churn Distribution

**Challenge:** Check whether the target variable is imbalanced (far more No than Yes).

**New techniques:** `value_counts().reindex()`, `plt.bar(colors)`

> Key takeaway: Always check target class balance first — severe imbalance determines whether you need stratified sampling, class weights, or resampling.

In [ ]:
plot_churn_distribution = __import__('6-plot_churn_distribution').plot_churn_distribution

plot_churn_distribution(df)

---
## Task 7 — Plot Categorical Distributions

**Challenge:** Visualize the frequency of every categorical feature in a single figure.

**New techniques:** `subplots(n,m)`, `flatten()`, `tick_params(rotation=45)`, `axis('off')`

> Key takeaway: A subplot grid is the standard pattern for batch-visualizing multiple features; `flatten()` + zip makes the loop trivial.

In [ ]:
plot_categorical_distributions = __import__('7-plot_categorical_distributions').plot_categorical_distributions

plot_categorical_distributions(df)

---
## Task 8 — Plot Continuous Distributions

**Challenge:** Show both the shape (histogram + KDE) and outliers (boxplot) of each numeric column in one figure.

**New techniques:** `gaussian_kde`, KDE overlay, `np.linspace`, `boxplot`

> Key takeaway: Histogram + KDE shows distribution shape; boxplot exposes outliers, skew, and quartiles. Together they give a complete picture of a continuous variable.

In [ ]:
plot_continuous_distributions = __import__('8-plot_continuous_distributions').plot_continuous_distributions

plot_continuous_distributions(df)

---
## Task 9 — Plot Correlation Heatmap

**Challenge:** Identify which numeric features move together, using a single visual.

**New techniques:** `df.corr()`, `sns.heatmap(annot=True, cmap='coolwarm')`

> Key takeaway: Correlation heatmaps quickly reveal multicollinearity — features that carry redundant information — critical for linear models.

In [ ]:
plot_correlation_heatmap = __import__('9-plot_correlation_heatmap').plot_correlation_heatmap

plot_correlation_heatmap(df)

---
## Task 10 — Plot Categorical vs Churn

**Challenge:** Quantify how churn rate varies across categories of a categorical feature.

**New techniques:** `groupby().apply(lambda)`, churn rate as boolean mean

> Key takeaway: `groupby + apply` is the pandas idiom for per-group statistics; `(x == 'Yes').mean()` directly gives the proportion without explicit counting.

In [ ]:
plot_categorical_vs_churn = __import__('10-plot_categorical_vs_churn').plot_categorical_vs_churn

plot_categorical_vs_churn(df, 'Contract')

---
## Task 11 — Plot Numeric vs Churn

**Challenge:** Compare the distribution of a continuous feature between churners and non-churners in a single plot — without overlapping.

**New techniques:** `pd.cut + unstack + plot.bar` → side-by-side grouped bar histogram

> Key takeaway: `pd.cut + unstack + plot.bar` turns a continuous-vs-categorical comparison into an intuitive side-by-side grouped bar chart — no manual bin shifting needed.

In [ ]:
plot_numeric_vs_churn = __import__('11-plot_numeric_vs_churn').plot_numeric_vs_churn

plot_numeric_vs_churn(df, 'tenure')

---
## Task 12 — Chi-Square Tests

**Challenge:** Quantify, with p-values, whether each categorical feature is statistically associated with Churn.

**New techniques:** `pd.crosstab`, `stats.chi2_contingency`

> Key takeaway: `crosstab + chi2_contingency` is the direct Python equivalent of Stata's `tabulate x y, chi2`.

In [ ]:
chi_square_tests = __import__('12-chi_square_tests').chi_square_tests

chi_results = chi_square_tests(df)
chi_df = (pd.Series(chi_results, name='p_value')
          .sort_values()
          .to_frame())
chi_df['significant'] = chi_df['p_value'] < 0.05
chi_df

---
## Task 13 — Welch's t-test

**Challenge:** Test whether the mean of each numeric feature differs significantly between churners and non-churners.

**New techniques:** `ttest_ind(equal_var=False)`, `select_dtypes(include='number')`

> Key takeaway: Welch's t-test is preferred when group variances are likely unequal; `equal_var=False` uses Welch-Satterthwaite df.

In [ ]:
ttest_numeric = __import__('13-ttest_numeric').ttest_numeric

ttest_results = ttest_numeric(df)
ttest_df = (pd.Series(ttest_results, name='p_value')
            .sort_values()
            .to_frame())
ttest_df['significant'] = ttest_df['p_value'] < 0.05
ttest_df

---
## Task 14 — Create Features

**Challenge:** Engineer two new features — `NumServices` (count of subscribed services) and `TenureGroup` (binned tenure). Then drop the source columns.

**New techniques:** `(df == val).sum(axis=1)`, `isin()`, `pd.cut` full params

> Key takeaway: Feature engineering transforms raw columns into more predictive representations — counting services reduces dimensionality; binning tenure captures non-linear relationships.

In [ ]:
create_features = __import__('14-create_features').create_features

columns_before = df.columns.tolist()
df = create_features(df)
columns_after = df.columns.tolist()

print('New columns:', set(columns_after) - set(columns_before))
print('Dropped columns:', set(columns_before) - set(columns_after))
print()
df[['NumServices', 'TenureGroup']].head(10)

---
## Task 15 — Encode Features

**Challenge:** Convert all categorical columns to numeric representations suitable for ML models.

**New techniques:** `LabelEncoder`, `OrdinalEncoder(categories=…)`, `get_dummies(drop_first=True, dtype=int)`

> Key takeaway: Three encoding strategies map to three column types: LabelEncoder for target, OrdinalEncoder for binary features, One-hot for nominal multi-category features.

In [ ]:
encode_features = __import__('15-encode_features').encode_features

df_enc, churn_le, binary_oe, tenure_oe = encode_features(df)

print('Encoded DataFrame columns:')
print(df_enc.columns.tolist())
print('\nEncoded types:')
print(df_enc.dtypes)
print('\nFirst 5 rows:')
df_enc.head()

---
## Task 16 — Scale Numeric

**Challenge:** Standardize numeric features (`MonthlyCharges`, `TotalCharges`) to mean=0, std=1 so they're on a common scale.

**New techniques:** `StandardScaler().fit_transform()`

> Key takeaway: Standardization (z-score) is essential for distance-sensitive models — without it, large-magnitude features dominate.

In [ ]:
scale_numeric = __import__('16-scale_numeric').scale_numeric

print('Before scaling:')
print(df_enc[['MonthlyCharges', 'TotalCharges']].describe().round(2))

df_enc = scale_numeric(df_enc)

print('\nAfter scaling (mean ~ 0, std ~ 1):')
print(df_enc[['MonthlyCharges', 'TotalCharges']].describe().round(4))

---
## Task 17 — Split Data

**Challenge:** Split data into train/test sets while preserving the class distribution of the target.

**New techniques:** `train_test_split(stratify=y)`

> Key takeaway: `stratify=y` ensures the train and test sets have the same target distribution — critical for imbalanced datasets.

In [ ]:
split_data = __import__('17-split_data').split_data

X_train, X_test, y_train, y_test = split_data(df_enc)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_test shape:  {y_test.shape}')
print()
print('Train Churn distribution:')
print(y_train.value_counts(normalize=True).round(3))
print()
print('Test Churn distribution:')
print(y_test.value_counts(normalize=True).round(3))

---
## Pipeline Complete

### What we built

| Phase | Tasks | Output |
|-------|-------|--------|
| EDA & Inspection | 0–1 | DataFrame structure, missingness plot |
| Data Cleaning | 2–5 | Cleaned DataFrame (correct types, no NaN, no duplicates, no IDs) |
| Visualization | 6–11 | 6 plots: target balance, categorical distributions, continuous distributions, correlation heatmap, churn rate by category, numeric-vs-churn side-by-side |
| Statistical Testing | 12–13 | Chi-square p-values per categorical feature, Welch's t-test p-values per numeric feature |
| Feature Engineering | 14–15 | NumServices, TenureGroup; all categoricals encoded to numeric |
| Scaling & Split | 16–17 | Standardized numeric features; stratified 80/20 train/test split |

**Ready for modeling:** `X_train`, `X_test`, `y_train`, `y_test` are fully preprocessed and ready for any scikit-learn classifier.